In [1]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score


In [2]:
data = {
    "ticket": [
        "I cannot login to my account",
        "Payment deducted but order not placed",
        "Website is very slow today",
        "Need refund for cancelled order",
        "Forgot my password and cannot access account",
        "App crashes when opening"
    ],

    "actual_tag": [
        "login issue",
        "payment problem",
        "slow performance",
        "refund request",
        "password reset",
        "technical bug"
    ]
}

df = pd.DataFrame(data)

print(df)

                                         ticket        actual_tag
0                  I cannot login to my account       login issue
1         Payment deducted but order not placed   payment problem
2                    Website is very slow today  slow performance
3               Need refund for cancelled order    refund request
4  Forgot my password and cannot access account    password reset
5                      App crashes when opening     technical bug


In [3]:
candidate_labels = [
    "login issue",
    "payment problem",
    "account access",
    "refund request",
    "technical bug",
    "delivery issue",
    "subscription",
    "password reset",
    "slow performance",
    "order cancellation"
]


In [4]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

C:\Users\HP\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
top_tags = []
top1_predictions = []

for text in df["ticket"]:

    result = classifier(
        text,
        candidate_labels,
        multi_label=True
    )

    # Top 3 tags
    tags = result["labels"][:3]

    top_tags.append(tags)

    # Top 1 prediction
    top1_predictions.append(tags[0])

# Add predictions to dataframe
df["top_3_tags"] = top_tags
df["predicted_tag"] = top1_predictions

In [6]:
accuracy = accuracy_score(
    df["actual_tag"],
    df["predicted_tag"]
)

print("\nAccuracy:", accuracy)


Accuracy: 0.6666666666666666


In [7]:
print("\nFinal Predictions:\n")

print(df[
    [
        "ticket",
        "actual_tag",
        "predicted_tag",
        "top_3_tags"
    ]
])


Final Predictions:

                                         ticket        actual_tag  \
0                  I cannot login to my account       login issue   
1         Payment deducted but order not placed   payment problem   
2                    Website is very slow today  slow performance   
3               Need refund for cancelled order    refund request   
4  Forgot my password and cannot access account    password reset   
5                      App crashes when opening     technical bug   

      predicted_tag                                         top_3_tags  
0       login issue    [login issue, account access, slow performance]  
1   payment problem  [payment problem, delivery issue, slow perform...  
2  slow performance     [slow performance, technical bug, login issue]  
3    refund request  [refund request, order cancellation, payment p...  
4       login issue      [login issue, password reset, account access]  
5  slow performance     [slow performance, technical bug,